# HomeGuardian - bounded Kaggle T4 experiment: media pipeline CPU vs GPU

Owner authorisation: **ONE** small T4 experiment, public/synthetic material only,
hard runtime budget **15 minutes**, no compilation.

The question is narrow: for the *development tooling* that processes demo video
(decode -> frames -> preprocessing -> OCR), does a GPU actually help, and by how much?
Tesseract OCR itself has no GPU path, so its cost is deliberately measured separately
rather than folded into a headline speedup.


In [ ]:
import os, json, time, shutil, subprocess, sys, glob

BUDGET_S = 15 * 60
T0 = time.time()
def elapsed():
    return round(time.time() - T0, 1)
def left():
    return round(BUDGET_S - elapsed(), 1)

def sh(cmd, timeout=None, check=False):
    t = time.time()
    try:
        p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True,
                           text=True, timeout=timeout)
        rc, out, err = p.returncode, (p.stdout or ""), (p.stderr or "")
    except subprocess.TimeoutExpired:
        rc, out, err = 124, "", "timeout after %ss" % timeout
    d = round(time.time() - t, 2)
    print("$ " + (cmd if isinstance(cmd, str) else " ".join(cmd)))
    print("  rc=%s  %.2fs" % (rc, d))
    if err.strip():
        print("  stderr: " + err.strip()[-600:])
    if check and rc != 0:
        raise SystemExit("FAILED: " + str(cmd))
    return rc, out, err, d

rep = {"schema": "hg-gpu-media-bench/1", "budget_seconds": BUDGET_S}
rep["gpu"] = {}
rc, out, _, _ = sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader")
rep["gpu"]["devices"] = out.strip().splitlines()
rc2, out2, _, _ = sh("nvidia-smi --query-gpu=count --format=csv,noheader")
rep["gpu"]["count"] = out2.strip().splitlines()
rep["cuda_visible"] = os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>")
rep["cpus"] = os.cpu_count()
print(json.dumps(rep["gpu"], indent=2))
print("elapsed %.1fs of %ss budget" % (elapsed(), BUDGET_S))


In [ ]:
# --- what media tooling is actually present? Capability first, benchmark second. ---
tools = {}
for b in ["ffmpeg", "ffprobe", "tesseract", "python3"]:
    tools[b] = shutil.which(b)
rc, out, _, _ = sh("ffmpeg -hide_banner -version") if tools["ffmpeg"] else (127, "", "", 0)
tools["ffmpeg_version"] = out.splitlines()[0] if out else None
rc, out, _, _ = sh("ffmpeg -hide_banner -hwaccels") if tools["ffmpeg"] else (127, "", "", 0)
tools["hwaccels"] = [l.strip() for l in out.splitlines()[1:] if l.strip()] if out else []
mods = {}
for m in ["torch", "cv2", "numpy", "cupy", "PIL", "pytesseract"]:
    try:
        __import__(m); mods[m] = True
    except Exception:
        mods[m] = False
tools["python_modules"] = mods
rep["tools"] = tools
print(json.dumps(tools, indent=2))
if mods.get("torch"):
    import torch
    rep["torch"] = {"version": torch.__version__, "cuda_available": torch.cuda.is_available(),
                    "device_count": torch.cuda.device_count(),
                    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    print("torch:", json.dumps(rep["torch"]))


In [ ]:
# --- SYNTHETIC material only: no downloads, no private media, no licensing questions ---
WORK = "/kaggle/working/bench"
os.makedirs(WORK, exist_ok=True)
SRC = os.path.join(WORK, "synthetic_source.mp4")
made = False
if tools["ffmpeg"]:
    rc, out, err, d = sh(
        f"ffmpeg -hide_banner -y -f lavfi -i testsrc2=size=1920x1080:rate=30:duration=60 "
        f"-f lavfi -i sine=frequency=440:duration=60 -vf \"drawtext=text='HOME GUARDIAN SYNTHETIC':"
        f"fontsize=64:x=60:y=60:fontcolor=white,drawtext=text='FRAME %{{n}}':fontsize=48:x=60:y=200:"
        f"fontcolor=yellow\" -c:v libx264 -preset ultrafast -crf 28 -c:a aac -shortest {SRC}", timeout=600)
    made = rc == 0 and os.path.exists(SRC)
rep["material"] = {"path": SRC, "created": made,
                   "size_bytes": os.path.getsize(SRC) if made else 0,
                   "synthetic": True, "source": "ffmpeg lavfi testsrc2 + drawtext (no external media)"}
print(json.dumps(rep["material"], indent=2))
print("elapsed %.1fs  budget left %.1fs" % (elapsed(), left()))
if not made:
    print("BLOCKED: cannot create synthetic source; remaining benchmarks will be skipped")


In [ ]:
# --- CPU decode + frame extraction (the baseline the 48-minute job actually paid) ---
NFRAMES = 120
cpu_frames = os.path.join(WORK, "frames_cpu")
shutil.rmtree(cpu_frames, ignore_errors=True); os.makedirs(cpu_frames, exist_ok=True)
cpu_dec = None
if made and left() > 240:
    cmd = (f"ffmpeg -hide_banner -nostdin -y -i {SRC} -vf fps=2 -frames:v {NFRAMES} "
           f"{cpu_frames}/f_%04d.png")
    rc, out, err, d = sh(cmd, timeout=420)
    got = len(glob.glob(cpu_frames + "/f_*.png"))
    cpu_dec = {"seconds": d, "frames": got, "rc": rc,
               "sec_per_frame": round(d / got, 4) if got else None, "mode": "cpu (libavcodec)"}
rep["cpu_decode"] = cpu_dec
print(json.dumps(cpu_dec, indent=2) if cpu_dec else "skipped (no material or budget)")
print("elapsed %.1fs  budget left %.1fs" % (elapsed(), left()))


In [ ]:
# --- GPU decode + extraction, with ACTUAL utilisation sampling (metadata is not proof) ---
import threading
usage = []
stop = threading.Event()
def sample():
    while not stop.is_set():
        r = subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits",
                           shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            usage.append(r.stdout.strip())
        time.sleep(1)
th = threading.Thread(target=sample, daemon=True); th.start()

gpu_frames = os.path.join(WORK, "frames_gpu")
shutil.rmtree(gpu_frames, ignore_errors=True); os.makedirs(gpu_frames, exist_ok=True)
gpu_dec = None
if made and left() > 240:
    attempts = []
    for name, vf, hw in [
        ("nvenc_scale_cuda", f"hwdownload,format=nv12,fps=2,scale=1920:1080", "-hwaccel cuda -hwaccel_output_format cuda"),
        ("cuda_upload_only",  f"fps=2", "-hwaccel cuda"),
    ]:
        if left() < 180:
            attempts.append({"variant": name, "skipped": "budget"}); continue
        shutil.rmtree(gpu_frames, ignore_errors=True); os.makedirs(gpu_frames, exist_ok=True)
        cmd = (f"ffmpeg -hide_banner -nostdin -y {hw} -i {SRC} -vf \"{vf}\" "
               f"-frames:v {NFRAMES} {gpu_frames}/g_%04d.png")
        rc, out, err, d = sh(cmd, timeout=420)
        got = len(glob.glob(gpu_frames + "/g_*.png"))
        attempts.append({"variant": name, "rc": rc, "seconds": d, "frames": got,
                         "sec_per_frame": round(d / got, 4) if got else None,
                         "ffmpeg_cmd": cmd})
        if rc == 0 and got > 0:
            break
    gpu_dec = {"attempts": attempts,
               "working_variant": next((a["variant"] for a in attempts if a.get("rc") == 0 and a.get("frames")), None)}
stop.set(); th.join(timeout=5)

def nums(x):
    return [int(s) for s in sh("", timeout=1)[0:0]] if not x else []
util = []
for line in usage:
    parts = [p.strip() for p in line.split(",")]
    if len(parts) >= 2 and parts[0].isdigit():
        util.append(int(parts[0]))
rep["gpu_decode"] = gpu_dec
rep["gpu_utilisation_during_decode"] = {
    "samples": len(util),
    "max_pct": max(util) if util else None,
    "mean_pct": round(sum(util) / len(util), 1) if util else None,
    "nonzero_samples": sum(1 for u in util if u > 0)}
print(json.dumps({k: rep[k] for k in ("gpu_decode", "gpu_utilisation_during_decode")}, indent=2))
print("elapsed %.1fs  budget left %.1fs" % (elapsed(), left()))


In [ ]:
# --- preprocessing (grayscale + threshold): the only in-pipeline op a GPU can accelerate ---
import numpy as np
frames = sorted(glob.glob(cpu_frames + "/f_*.png"))[:60]
pre = {}
if frames and mods.get("cv2"):
    import cv2
    t = time.time()
    acc = 0
    for f in frames:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        _, th = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
        acc += int(th.sum())
    pre["cpu_cv2"] = {"seconds": round(time.time() - t, 3), "frames": len(frames),
                      "checksum": acc}
if frames and mods.get("torch"):
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    t = time.time(); acc = 0
    for f in frames:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE) if mods.get("cv2") else None
        if img is None:
            break
        x = torch.from_numpy(img).to(dev)
        th = (x > 127).to(torch.uint8) * 255
        acc += int(th.sum().item())
    pre["torch_" + dev] = {"seconds": round(time.time() - t, 3), "frames": len(frames),
                           "checksum": acc, "device": dev}
    if dev == "cuda":
        try:
            torch.cuda.synchronize()
            pre["torch_cuda_mem_mb"] = round(torch.cuda.max_memory_allocated() / 2**20, 1)
        except Exception:
            pass
rep["preprocessing"] = pre
print(json.dumps(pre, indent=2) if pre else "skipped (no frames or no cv2/torch)")
print("elapsed %.1fs  budget left %.1fs" % (elapsed(), left()))


In [ ]:
# --- OCR: measured SEPARATELY because Tesseract has no GPU path ---
ocr = {"note": "tesseract has no CUDA implementation; this cost cannot be GPU-accelerated",
       "gpu_ocr_available": False}
K = 15
sel = frames[:K]
if sel and tools["tesseract"]:
    t = time.time(); txt = ""
    for f in sel:
        r = subprocess.run(["tesseract", f, "stdout"], capture_output=True, text=True)
        txt += r.stdout
    ocr["cpu_tesseract"] = {"seconds": round(time.time() - t, 3), "frames": len(sel),
                            "sec_per_frame": round((time.time() - t) / len(sel), 3),
                            "text_chars": len(txt.strip()),
                            "sample_text": txt.strip()[:120]}
elif sel and mods.get("pytesseract"):
    ocr["cpu_pytesseract"] = "module present but tesseract binary missing"
rep["ocr"] = ocr
print(json.dumps(ocr, indent=2))
print("elapsed %.1fs  budget left %.1fs" % (elapsed(), left()))


In [ ]:
# --- consolidate: speedups, utilisation, and an honest statement of what was NOT accelerated ---
c = rep.get("cpu_decode") or {}
g = rep.get("gpu_decode") or {}
gv = g.get("working_variant")
ga = next((a for a in g.get("attempts", []) if a.get("variant") == gv), {}) if gv else {}
if c.get("sec_per_frame") and ga.get("sec_per_frame"):
    rep["decode_speedup_gpu_over_cpu"] = round(c["sec_per_frame"] / ga["sec_per_frame"], 3)
pre = rep.get("preprocessing", {})
if pre.get("cpu_cv2") and pre.get("torch_cuda"):
    rep["preproc_speedup_gpu_over_cpu"] = round(pre["cpu_cv2"]["seconds"] / pre["torch_cuda"]["seconds"], 3)
rep["elapsed_seconds"] = elapsed()
rep["within_budget"] = elapsed() <= BUDGET_S
rep["local_workstation_impact"] = "none - entire benchmark executed on the remote runner"
open("/kaggle/working/gpu_bench_report.json", "w").write(json.dumps(rep, indent=2))
print(json.dumps({k: rep.get(k) for k in
                  ("decode_speedup_gpu_over_cpu", "preproc_speedup_gpu_over_cpu",
                   "gpu_utilisation_during_decode", "elapsed_seconds", "within_budget")}, indent=2))
print("\nGPU ACTUALLY USED:", rep["gpu"].get("devices"), "| count:", rep["gpu"].get("count"))
print("OCR GPU-accelerable:", rep.get("ocr", {}).get("gpu_ocr_available"))
print("report written to /kaggle/working/gpu_bench_report.json")
